In [1]:
pip install geopandas shapely folium pandas

In [3]:
import geopandas as gpd
from shapely.geometry import Point

# A. Data Gerai Eksisting (Store_A & Store_B)
existing_stores_data = {
    'store_id': ['Store_A', 'Store_B'],
    'geometry': [
        Point(102.265, -3.795),
        Point(102.280, -3.805)
    ]
}
gdf_existing = gpd.GeoDataFrame(existing_stores_data, crs="EPSG:4326")

# B. Data Kandidat Lokasi Baru (Site_1 s/d Site_4)
candidates_data = {
    'site_id': ['Site_1', 'Site_2', 'Site_3', 'Site_4'],
    'est_population': [2500, 4200, 1800, 5000],  # Proyeksi jumlah populasi
    'geometry': [
        Point(102.268, -3.797),
        Point(102.290, -3.810),
        Point(102.250, -3.780),
        Point(102.275, -3.820)
    ]
}
gdf_candidates = gpd.GeoDataFrame(candidates_data, crs="EPSG:4326")

# Reproyeksi ke UTM Zone 48S (EPSG:32748) khusus wilayah Kota Bengkulu agar jarak akurat dalam meter
gdf_existing_proj = gdf_existing.to_crs(epsg=32748)
gdf_candidates_proj = gdf_candidates.to_crs(epsg=32748)

print("Data spasial gerai eksisting dan kandidat lokasi berhasil disiapkan!")

Data spasial gerai eksisting dan kandidat lokasi berhasil disiapkan!


In [4]:
# 1. Buat zona penyangga (Buffer) 1.000 meter di sekitar gerai eksisting
buffer_radius_meters = 1000
gdf_existing_proj['buffer'] = gdf_existing_proj.geometry.buffer(buffer_radius_meters)

# Gabungkan poligon buffer menjadi satu area terlarang (Cannibalization Zone)
cannibalization_zone = gdf_existing_proj['buffer'].unary_union

# 2. Evaluasi kandidat lokasi berbasis aturan bisnis
results = []
min_population_threshold = 2000  # Minimal populasi 2.000 jiwa

for idx, row in gdf_candidates_proj.iterrows():
    # Cek apakah lokasi berada di dalam area kanibalisasi
    is_inside_buffer = row.geometry.within(cannibalization_zone)

    # Cek kriteria populasi
    has_enough_pop = row['est_population'] >= min_population_threshold

    # Keputusan Akhir
    if not is_inside_buffer and has_enough_pop:
        status = 'REKOMENDASI'
    elif is_inside_buffer:
        status = 'DITOLAK (Terlalu Dekat Gerai Lain)'
    else:
        status = 'DITOLAK (Populasi Rendah)'

    results.append({
        'site_id': row['site_id'],
        'est_population': row['est_population'],
        'status': status,
        'geometry': row.geometry
    })

# 3. Simpan hasil ke GeoDataFrame dan kembalikan ke sistem koordinat EPSG:4326
gdf_results_proj = gpd.GeoDataFrame(results, crs=gdf_candidates_proj.crs)
gdf_results = gdf_results_proj.to_crs(epsg=4326)

# Tampilkan Tabel Hasil
print("=== HASIL EVALUASI LOKASI RETAIL ===")
gdf_results[['site_id', 'est_population', 'status']]

=== HASIL EVALUASI LOKASI RETAIL ===


/tmp/ipykernel_7300/2776483048.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cannibalization_zone = gdf_existing_proj['buffer'].unary_union


,site_id,est_population,status
0,Site_1,2500,DITOLAK (Terlalu Dekat Gerai Lain)
1,Site_2,4200,REKOMENDASI
2,Site_3,1800,DITOLAK (Populasi Rendah)
3,Site_4,5000,REKOMENDASI


In [8]:
import folium

# Inisialisasi Peta menggunakan tiles Esri WorldStreetMap (Gratis & Tanpa API Key)
m = folium.Map(
    location=[-3.800, 102.275],
    zoom_start=13,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

# 1. Plot Gerai Eksisting & Lingkaran Zona Kanibalisasi (1 KM)
for idx, row in gdf_existing.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"<b>Eksisting:</b> {row['store_id']}",
        icon=folium.Icon(color='blue', icon='shopping-cart')
    ).add_to(m)

    folium.Circle(
        location=[row.geometry.y, row.geometry.x],
        radius=1000,
        color='red',
        fill=True,
        fill_opacity=0.15,
        popup="Zona Kanibalisasi (1 km)"
    ).add_to(m)

# 2. Plot Kandidat Lokasi Berdasarkan Hasil Evaluasi
for idx, row in gdf_results.iterrows():
    if row['status'] == 'REKOMENDASI':
        color = 'green'
        icon = 'star'
    else:
        color = 'gray'
        icon = 'info-sign'

    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"<b>{row['site_id']}</b><br>Populasi: {row['est_population']}<br>Status: {row['status']}",
        icon=folium.Icon(color=color, icon=icon)
    ).add_to(m)

# Tampilkan Peta Interaktif
m